# Label 4 slow order lines (answer key for the `case` questions)

For each line: read the timeline and the peer comparison, then write **one sentence**: where the time went (between which two steps, how many days) and whether that is **abnormal** or **normal** for this vendor / similar lines.
Only claim what the events show, not why a person did it.

In [7]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 60)
con = duckdb.connect("../data/p2p.duckdb", read_only=True)


def timeline(case_id: str) -> pd.DataFrame:
    return con.execute(
        """
        SELECT seq, activity, ts, ts_status, user_id, value_eur, n_identical,
               round(date_diff('second', lag(ts) OVER (ORDER BY seq), ts) / 86400.0, 1) AS wait_days
        FROM events WHERE case_id = ? AND is_canonical ORDER BY seq""",
        [case_id],
    ).df()


con.execute(
    "SELECT * FROM cases WHERE case_id IN "
    "('4507000365_00010', '4507000414_00010', '4507000905_00050', '4507004456_00010')"
).df()

,case_id,po,item,item_category,item_type,vendor,company,spend_area,sub_spend_area,document_type,gr_based_inv_verif,goods_receipt_flag,value_eur,n_events,n_bad_dates,start_ts,end_ts,throughput_days
0,4507000365_00010,4507000365,00010,"3-way match, invoice before GR",Standard,vendorID_0175,companyID_0000,Packaging,Metal Containers & Lids < 30L,Standard PO,False,True,4140.0,6,0,2018-01-02 11:47:00,2018-05-24 11:32:00,141.99
1,4507000414_00010,4507000414,00010,"3-way match, invoice before GR",Standard,vendorID_0195,companyID_0000,Packaging,Metal Containers & Lids < 30L,Standard PO,False,True,1703.0,6,0,2018-01-02 13:49:00,2018-05-04 08:14:00,121.77
2,4507000905_00050,4507000905,00050,"3-way match, invoice before GR",Standard,vendorID_0167,companyID_0000,Packaging,Metal Containers & Lids < 30L,Standard PO,False,True,2940.0,6,0,2018-01-04 09:19:00,2018-05-31 10:32:00,147.05
3,4507004456_00010,4507004456,00010,"3-way match, invoice before GR",Standard,vendorID_0199,companyID_0000,Sales,Products for Resale,Standard PO,False,True,1773.0,7,0,2018-01-22 11:12:00,2018-06-07 11:11:00,136.00


## Is each wait normal? Compare with peers

Each wait is compared with the **same step** (same previous activity → same activity) in two peer groups:
- **vendor**: all lines of the same vendor
- **similar**: lines with the same item category and value band (5 equal-size bands by `value_eur`)

`*_pctile` = % of peers that waited less than this line. ~50 = typical, 90+ = unusually slow, below 25 = fast.
`n_*` = how many peers; under ~30, treat the comparison as weak.

In [8]:
# Peer table: every step of every line, with its wait, vendor, category and value band
con.execute("""
    CREATE OR REPLACE TEMP TABLE peers AS
    WITH s AS (
        SELECT case_id, seq, lag(activity) OVER w AS prev_activity, activity,
               date_diff('second', lag(ts) OVER w, ts) / 86400.0 AS wait_days
        FROM events WHERE is_canonical AND seq IS NOT NULL AND ts_status IN ('ok', 'repaired')
        WINDOW w AS (PARTITION BY case_id ORDER BY seq))
    SELECT s.*, c.vendor, c.item_category, ntile(5) OVER (ORDER BY c.value_eur) AS value_band
    FROM s JOIN cases c USING (case_id) WHERE s.prev_activity IS NOT NULL""")


def benchmark(case_id: str) -> pd.DataFrame:
    """Each wait vs the same step at the same vendor, and at lines of the same category and value band."""
    return con.execute(
        """
        WITH me AS (SELECT * FROM peers WHERE case_id = ?),
        cmp AS (
            SELECT me.seq, me.prev_activity || ' -> ' || me.activity AS step, me.wait_days AS wait,
                   p.wait_days AS peer_wait,
                   p.vendor = me.vendor AS same_vendor,
                   p.item_category = me.item_category AND p.value_band = me.value_band AS similar_line
            FROM me JOIN peers p ON p.prev_activity = me.prev_activity AND p.activity = me.activity)
        SELECT seq, step, round(any_value(wait), 1) AS wait,
               count(*) FILTER (WHERE same_vendor) AS n_vendor,
               round(median(peer_wait) FILTER (WHERE same_vendor), 1) AS vendor_median,
               round(100 * avg((peer_wait < wait)::int) FILTER (WHERE same_vendor)) AS vendor_pctile,
               count(*) FILTER (WHERE similar_line) AS n_similar,
               round(median(peer_wait) FILTER (WHERE similar_line), 1) AS similar_median,
               round(100 * avg((peer_wait < wait)::int) FILTER (WHERE similar_line)) AS similar_pctile
        FROM cmp GROUP BY seq, step ORDER BY seq""",
        [case_id],
    ).df()

In [9]:
display(timeline("4507000365_00010"))
benchmark("4507000365_00010")

,seq,activity,ts,ts_status,user_id,value_eur,n_identical,wait_days
0,1,Create Purchase Order Item,2018-01-02 11:47:00,ok,user_033,4140.0,1,NaN
1,2,Vendor creates invoice,2018-02-13 22:59:00,ok,None,4140.0,1,42.5
2,3,Record Goods Receipt,2018-02-14 08:41:00,ok,user_034,4140.0,1,0.4
3,4,Record Invoice Receipt,2018-02-15 14:21:00,ok,user_012,4140.0,1,1.2
4,5,Remove Payment Block,2018-03-27 15:29:00,ok,user_006,4140.0,1,40.0
5,6,Clear Invoice,2018-05-24 11:32:00,ok,user_002,4140.0,1,57.8


,seq,step,wait,n_vendor,vendor_median,vendor_pctile,n_similar,similar_median,similar_pctile
0,2,Create Purchase Order Item -> Vendor creates i...,42.5,570,21.5,86.0,14041,15.6,87.0
1,3,Vendor creates invoice -> Record Goods Receipt,0.4,676,0.6,12.0,18464,1.6,16.0
2,4,Record Goods Receipt -> Record Invoice Receipt,1.2,718,3.1,25.0,18396,11.9,14.0
3,5,Record Invoice Receipt -> Remove Payment Block,40.0,144,6.3,90.0,5825,8.0,79.0
4,6,Remove Payment Block -> Clear Invoice,57.8,234,57.7,50.0,8731,30.9,74.0


**case_01 label:** 
- The time between purchasing the order and the vendor creating the invoice is of 42.5 days, which is rare for this vendor since it is in the 86.0 percentile of waiting time for this step (median of 21.5). Moreover, comapred to other vendors that sell items of similar monetary value, this waiting time would be in the 87% percentile.
- Similarly, the wait time between the record invoice receipt and removing the payment block was of 40 days, which is in the 90th percentile for this vendor and 79% percentile for items of the similar price of all vendors. This vendor was actually historically fast for this step, with a median of 6.3 waiting days, so this could have prolongated the overall waitimng time significantly.
- Finally, the waiting time between the removal of the payment block and the clearance of the invoice was 57.8 days, which is exacly the median for this vendor (50.0% percentile). It must be said that this waiting time is actually quite big in comparison to other vendors selling items of similar value (74% percentile).
- The combination of these three factors led to a slow order.

In [10]:
display(timeline("4507000414_00010"))
benchmark("4507000414_00010")

,seq,activity,ts,ts_status,user_id,value_eur,n_identical,wait_days
0,1,Create Purchase Order Item,2018-01-02 13:49:00,ok,user_108,1703.0,1,NaN
1,2,Record Goods Receipt,2018-01-17 13:42:00,ok,user_077,1703.0,1,15.0
2,3,Vendor creates invoice,2018-01-29 22:59:00,ok,None,1703.0,1,12.4
3,4,Record Invoice Receipt,2018-04-26 15:32:00,ok,user_004,1703.0,1,86.7
4,5,Remove Payment Block,2018-04-28 06:43:00,ok,user_006,1703.0,1,1.6
5,6,Clear Invoice,2018-05-04 08:14:00,ok,user_002,1703.0,1,6.1


,seq,step,wait,n_vendor,vendor_median,vendor_pctile,n_similar,similar_median,similar_pctile
0,2,Create Purchase Order Item -> Record Goods Rec...,15.0,260,30.5,18.0,11609,13.9,56.0
1,3,Record Goods Receipt -> Vendor creates invoice,12.4,254,3.3,89.0,17734,0.6,93.0
2,4,Vendor creates invoice -> Record Invoice Receipt,86.7,248,7.6,96.0,19777,5.4,99.0
3,5,Record Invoice Receipt -> Remove Payment Block,1.6,88,3.4,5.0,9129,10.2,13.0
4,6,Remove Payment Block -> Clear Invoice,6.1,99,26.9,21.0,11091,23.0,23.0


**case_02 label:** 

In [11]:
display(timeline("4507000905_00050"))
benchmark("4507000905_00050")

,seq,activity,ts,ts_status,user_id,value_eur,n_identical,wait_days
0,1,Create Purchase Order Item,2018-01-04 09:19:00,ok,user_142,2940.0,1,NaN
1,2,Change Quantity,2018-01-19 09:39:00,ok,user_142,2940.0,1,15.0
2,3,Vendor creates invoice,2018-02-05 22:59:00,ok,None,2940.0,1,17.6
3,4,Record Goods Receipt,2018-02-06 12:35:00,ok,user_080,2940.0,1,0.6
4,5,Record Invoice Receipt,2018-05-30 15:29:00,ok,user_013,2940.0,1,113.1
5,6,Clear Invoice,2018-05-31 10:32:00,ok,user_002,2940.0,1,0.8


,seq,step,wait,n_vendor,vendor_median,vendor_pctile,n_similar,similar_median,similar_pctile
0,2,Create Purchase Order Item -> Change Quantity,15.0,375,9.2,68.0,3688,8.1,63.0
1,3,Change Quantity -> Vendor creates invoice,17.6,88,20.5,44.0,2013,6.5,74.0
2,4,Vendor creates invoice -> Record Goods Receipt,0.6,324,2.4,40.0,22231,1.5,30.0
3,5,Record Goods Receipt -> Record Invoice Receipt,113.1,357,35.0,97.0,22945,10.9,98.0
4,6,Record Invoice Receipt -> Clear Invoice,0.8,649,51.1,4.0,26940,42.0,2.0


**case_03 label:** 

In [12]:
display(timeline("4507004456_00010"))
benchmark("4507004456_00010")

,seq,activity,ts,ts_status,user_id,value_eur,n_identical,wait_days
0,1,Create Purchase Order Item,2018-01-22 11:12:00,ok,user_089,1773.0,1,NaN
1,2,Vendor creates invoice,2018-01-26 22:59:00,ok,None,1773.0,1,4.5
2,3,Record Goods Receipt,2018-01-29 09:39:00,ok,batch_03,1773.0,1,2.4
3,4,Record Invoice Receipt,2018-02-01 10:51:00,ok,user_018,1773.0,1,3.1
4,5,Change Price,2018-05-16 13:52:00,ok,user_089,1773.0,1,104.1
5,6,Remove Payment Block,2018-06-05 11:09:00,ok,user_112,1773.0,1,19.9
6,7,Clear Invoice,2018-06-07 11:11:00,ok,user_002,1773.0,1,2.0


,seq,step,wait,n_vendor,vendor_median,vendor_pctile,n_similar,similar_median,similar_pctile
0,2,Create Purchase Order Item -> Vendor creates i...,4.5,80,6.3,10.0,20334,11.6,25.0
1,3,Vendor creates invoice -> Record Goods Receipt,2.4,81,2.4,63.0,22231,1.5,63.0
2,4,Record Goods Receipt -> Record Invoice Receipt,3.1,102,4.9,35.0,22945,10.9,24.0
3,5,Record Invoice Receipt -> Change Price,104.1,10,60.2,70.0,435,6.9,95.0
4,6,Change Price -> Remove Payment Block,19.9,12,0.3,92.0,389,0.0,97.0
5,7,Remove Payment Block -> Clear Invoice,2.0,49,6.6,22.0,11091,23.0,15.0


**case_04 label:** 